# Frozen-checkpoint CODI answer-colon inference ablation

This notebook tests whether the energy, answer-aware, or parameter-aware residual directions are causally necessary for CODI's held-out accuracy. It never trains the model. It replaces candidate coordinates by their fresh student mean only on the forward pass that consumes the generated `The answer is:` colon, compares every result to the same frozen baseline, and uses rank-matched random ablations as the null. Enable Internet and a T4-or-newer GPU, attach the three completed selector datasets, then use **Save Version → Save & Run All**.

## 1. Frozen configuration

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the immutable commit after pushing.
REPO_DIR = "/kaggle/working/latent-reasoning"
REPRODUCTION_SUMMARY_INPUT = ""
RESUME_INPUT = ""  # Optional previous export root.
ENERGY_BASIS_INPUT = ""
ANSWER_CONDITIONED_BASIS_INPUT = ""
PARAMETER_AWARE_BASIS_INPUT = ""
RUN_REPRODUCTION_GATE_IF_MISSING = True
RUN_SMOKE = True
RUN_FULL = True
CALIBRATION_EXAMPLES = 1024
CALIBRATION_SEED = 67
CALIBRATION_BATCH_SIZE = 16
RANDOM_REPLICATES = 20
RANDOM_SEED = 20260806
EVAL_BATCH_SIZE = 32
PRECISION = "auto"
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 0
FAMILYWISE_ALPHA = 0.05
UPLOAD_AS_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = "jonraza15/official-codi-endpoint-inference-ablation"

## 2. Install, pin, and test the implementation

In [ ]:
import datetime, hashlib, json, os, pathlib, shutil, subprocess, sys
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception: pass
repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir(): subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main": print("PIN RUN_COMMIT BEFORE THE FINAL RUN:", commit)
import torch, transformers
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
print("Torch:", torch.__version__, "Transformers:", transformers.__version__, "GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_endpoint_inference_ablation.py", "tests/test_endpoint_retention.py", "tests/test_official_codi_target_utility.py"], cwd=REPO_DIR, check=True)

## 3. Locate the three immutable completed selector artifacts

In [ ]:
EXPECTED_CONTRACTS = {"source_faithful_student_and_teacher_answer_colon_v2": "energy", "answer_conditioned_colon_block_states_v1": "answer_conditioned", "parameter_aware_colon_final_two_blocks_v1": "parameter_aware"}
EXPLICIT = {"energy": ENERGY_BASIS_INPUT, "answer_conditioned": ANSWER_CONDITIONED_BASIS_INPUT, "parameter_aware": PARAMETER_AWARE_BASIS_INPUT}
def merged_metadata(path, payload):
    metadata = dict(payload.get("metadata", {})); manifest = path.parent / "run_manifest.json"; parity = path.parent / "native_loss_gradient_parity.json"
    if manifest.is_file():
        for key, value in json.loads(manifest.read_text()).items(): metadata.setdefault(key, value)
    if parity.is_file() and "native_parity_gate" not in metadata: metadata["native_parity_gate"] = json.loads(parity.read_text())
    return metadata
found = {method: [] for method in EXPECTED_CONTRACTS.values()}; diagnostics = []
for path in pathlib.Path("/kaggle/input").rglob("basis.pt"):
    try: payload = torch.load(path, map_location="cpu", weights_only=False); metadata = merged_metadata(path, payload)
    except Exception as error: diagnostics.append((str(path), type(error).__name__)); continue
    contract = metadata.get("contract"); diagnostics.append((str(path), contract, metadata.get("calibration_examples"), metadata.get("residual_fit_examples")))
    if contract in EXPECTED_CONTRACTS:
        method = EXPECTED_CONTRACTS[contract]
        full = (method == "energy" and metadata.get("calibration_examples") == 5000) or (method != "energy" and metadata.get("residual_fit_examples") == 1024 and metadata.get("direction_selection_examples") == 1024)
        if full and metadata.get("native_parity_gate", {}).get("status") == "passed": found[method].append(path)
basis_by_method = {}
for method, explicit in EXPLICIT.items():
    paths = [pathlib.Path(explicit)] if explicit else found[method]
    if not explicit and paths:
        by_sha = {}
        for path in paths: by_sha.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), []).append(path)
        if len(by_sha) == 1: paths = [sorted(next(iter(by_sha.values())), key=lambda p: (len(p.parts), p.as_posix()))[0]]
    if len(paths) != 1:
        print(*diagnostics, sep="\n"); raise AssertionError(f"Need exactly one completed {method} basis; set its explicit path if necessary. Found {paths}")
    basis_by_method[method] = paths[0]
ENERGY_BASIS = basis_by_method["energy"]; ANSWER_CONDITIONED_BASIS = basis_by_method["answer_conditioned"]; PARAMETER_AWARE_BASIS = basis_by_method["parameter_aware"]
for method, path in basis_by_method.items(): print(method, path)

## 4. Durable paths, logging, resume, and reproduction gate

In [ ]:
OUTPUT_ROOT = repo / "outputs" / "official_codi_endpoint_inference_ablation"
RUNS_ROOT = OUTPUT_ROOT / "runs"; STATS_ROOT = OUTPUT_ROOT / "activation_stats_seed67"
REPORT_ROOT = repo / "reports" / "official_codi_endpoint_inference_ablation"
LOG_ROOT = repo / "logs" / "official_codi_endpoint_inference_ablation"
VALIDATION_ROOT = repo / "outputs" / "official_codi_gpt2"
for path in (RUNS_ROOT, STATS_ROOT, REPORT_ROOT, LOG_ROOT, VALIDATION_ROOT): path.mkdir(parents=True, exist_ok=True)
def run_persisted(command, log_name):
    log_path = LOG_ROOT / log_name; print("Starting:", " ".join(map(str, command)), flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout: print(line, end="", flush=True); log.write(line)
        code = process.wait()
    if code != 0: raise RuntimeError(f"Command failed with exit code {code}; inspect {log_path}")
if RESUME_INPUT:
    candidates = [p for p in pathlib.Path(RESUME_INPUT).rglob("official_codi_endpoint_inference_ablation") if p.is_dir()]
    assert candidates; shutil.copytree(sorted(candidates, key=lambda p: (len(p.parts), p.as_posix()))[0], OUTPUT_ROOT, dirs_exist_ok=True)
EXPECTED_REVISION = "fd641b3d3edc59e4f534b55588e906588c9e36bb"
def passed_summary(path):
    try: payload = json.loads(path.read_text())
    except Exception: return False
    gate = payload.get("accuracy_gate", payload.get("gate")); status = gate.get("status") if isinstance(gate, dict) else gate
    return status == "passed" and payload.get("evaluated_counts", {}).get("gsm8k") == 1319 and payload.get("checkpoint_revision") in {None, EXPECTED_REVISION}
if REPRODUCTION_SUMMARY_INPUT:
    REPRODUCTION_SUMMARY = pathlib.Path(REPRODUCTION_SUMMARY_INPUT); assert passed_summary(REPRODUCTION_SUMMARY)
else:
    candidates = [p for p in pathlib.Path("/kaggle/input").rglob("summary.json") if passed_summary(p)] + [p for p in VALIDATION_ROOT.rglob("summary.json") if passed_summary(p)]
    if not candidates:
        assert RUN_REPRODUCTION_GATE_IF_MISSING
        run_persisted([sys.executable, "-u", "-m", "src.eval.official_codi", "--config", "configs/official_codi_gpt2.yaml", "--datasets", "gsm8k", "--limit", "0", "--device", "cuda", "--output-dir", str(VALIDATION_ROOT)], "official_codi_gsm8k_gate.log")
        candidates = [p for p in VALIDATION_ROOT.rglob("summary.json") if passed_summary(p)]
    assert candidates; REPRODUCTION_SUMMARY = sorted(candidates, key=lambda p: p.as_posix())[0]
print("Reproduction summary:", REPRODUCTION_SUMMARY)

## 5. Fit the frozen student endpoint mean on fresh questions

In [ ]:
STATS_PATH = STATS_ROOT / "activation_stats.pt"
stats_command = [sys.executable, "-u", "scripts/collect_official_codi_endpoint_activation_stats.py", "--config", "configs/official_codi_gpt2.yaml", "--reproduction-summary", str(REPRODUCTION_SUMMARY), "--energy-basis", str(ENERGY_BASIS), "--answer-conditioned-basis", str(ANSWER_CONDITIONED_BASIS), "--parameter-aware-basis", str(PARAMETER_AWARE_BASIS), "--output-dir", str(STATS_ROOT), "--examples", str(CALIBRATION_EXAMPLES), "--batch-size", str(CALIBRATION_BATCH_SIZE), "--sampling-seed", str(CALIBRATION_SEED), "--precision", "float32", "--device", "cuda"]
run_persisted(stats_command, "collect_activation_stats.log")
stats = torch.load(STATS_PATH, map_location="cpu", weights_only=False)
assert stats["count"] == CALIBRATION_EXAMPLES and stats["student_mean"].shape == (13, 768)
print("Fresh calibration request:", stats["request_sha256"])

## 6. Register the 82 paired causal arms

In [ ]:
METHODS = ["energy", "answer_conditioned", "parameter_aware"]
CANDIDATE_ARMS = [f"remove_{method}_joint" for method in METHODS] + [f"remove_{method}_s{state}_d{slot}" for method in METHODS for state in (11, 12) for slot in range(3)]
RANDOM_ARMS = [f"remove_random_joint_r{replicate:02d}" for replicate in range(RANDOM_REPLICATES)] + [f"remove_random_s{state}_r{replicate:02d}" for state in (11, 12) for replicate in range(RANDOM_REPLICATES)]
ARMS = ["baseline", *CANDIDATE_ARMS, *RANDOM_ARMS]
assert len(ARMS) == 82 and len(set(ARMS)) == len(ARMS)
print("Candidate arms:", len(CANDIDATE_ARMS), "random-null arms:", len(RANDOM_ARMS), "total:", len(ARMS))

## 7. Smoke-test exact cue tracking and the causal hook

In [ ]:
def ablation_command(root, arm, eval_limit=0):
    return [sys.executable, "-u", "scripts/run_official_codi_endpoint_inference_ablation.py", "--config", "configs/official_codi_gpt2.yaml", "--reproduction-summary", str(REPRODUCTION_SUMMARY), "--energy-basis", str(ENERGY_BASIS), "--answer-conditioned-basis", str(ANSWER_CONDITIONED_BASIS), "--parameter-aware-basis", str(PARAMETER_AWARE_BASIS), "--activation-stats", str(STATS_PATH), "--output-dir", str(root), "--arm", arm, "--random-replicates", str(RANDOM_REPLICATES), "--random-seed", str(RANDOM_SEED), "--alpha", "1.0", "--eval-limit", str(eval_limit), "--eval-batch-size", str(EVAL_BATCH_SIZE), "--precision", PRECISION, "--device", "cuda"]
if RUN_SMOKE:
    smoke_reach = []
    for arm in ("baseline", "remove_answer_conditioned_joint"):
        root = OUTPUT_ROOT / "smoke" / arm; run_persisted(ablation_command(root, arm, eval_limit=32), f"smoke_{arm}.log")
        summary = json.loads((root / "summary.json").read_text()); smoke_reach.append(summary["endpoint_coverage"]["endpoint_reached_count"])
    assert smoke_reach[0] == smoke_reach[1] and smoke_reach[0] >= 31, "Cue reach must be paired and at least 95%"
    print("Smoke cue reach:", smoke_reach[0], "/ 32")

## 8. Run every frozen arm on full GSM8K

In [ ]:
if RUN_FULL:
    for arm in ARMS:
        run_persisted(ablation_command(RUNS_ROOT / arm, arm), f"{arm}.log")
summaries = list(RUNS_ROOT.rglob("summary.json"))
print("Completed full arms:", len(summaries), "/", len(ARMS))
if RUN_FULL: assert len(summaries) == len(ARMS)

## 9. Paired confirmatory analysis

In [ ]:
REPORT_PATH = REPORT_ROOT / "endpoint_inference_ablation_summary.json"
if RUN_FULL:
    run_persisted([sys.executable, "-u", "scripts/analyze_official_codi_endpoint_inference_ablation.py", "--runs-root", str(RUNS_ROOT), "--output", str(REPORT_PATH), "--bootstrap-samples", str(BOOTSTRAP_SAMPLES), "--bootstrap-seed", str(BOOTSTRAP_SEED), "--familywise-alpha", str(FAMILYWISE_ALPHA)], "analyze_endpoint_inference_ablation.log")
    report = json.loads(REPORT_PATH.read_text())
    print("Baseline accuracy:", report["baseline_accuracy"], "cue coverage:", report["answer_cue_endpoint_coverage"])
    print("Accuracy-critical directions/groups:", report["accuracy_critical_directions_or_groups"])
    candidates = [(name, value) for name, value in report["comparisons"].items() if value.get("spec", {}).get("family") in {"selected_joint", "selected_single"}]
    for name, value in sorted(candidates, key=lambda item: item[1]["accuracy_loss"], reverse=True):
        print(name, "PC=", value["spec"].get("residual_pc_index"), "loss_pp=", round(value["accuracy_loss_percentage_points"], 4), "CI=", value["bootstrap_95_ci"], "Holm p=", value["holm_adjusted_mcnemar_p"], "random p=", value["empirical_random_null_p"], "critical=", value["accuracy_critical"])

## 10. Export checksummed, resumable results

In [ ]:
EXPORT_ROOT = pathlib.Path("/kaggle/working/official_codi_endpoint_inference_ablation_export")
if EXPORT_ROOT.exists(): shutil.rmtree(EXPORT_ROOT)
export_repo = EXPORT_ROOT / "latent-reasoning"
shutil.copytree(OUTPUT_ROOT, export_repo / "outputs" / "official_codi_endpoint_inference_ablation")
if REPORT_ROOT.exists(): shutil.copytree(REPORT_ROOT, export_repo / "reports" / "official_codi_endpoint_inference_ablation")
shutil.copytree(LOG_ROOT, export_repo / "logs" / "official_codi_endpoint_inference_ablation")
validation = export_repo / "outputs" / "official_codi_gpt2_reproduction"; validation.mkdir(parents=True, exist_ok=True); shutil.copy2(REPRODUCTION_SUMMARY, validation / "summary.json")
(EXPORT_ROOT / "RUN_COMMIT.txt").write_text(commit + "\n")
(EXPORT_ROOT / "RUN_INSTRUCTIONS.txt").write_text("Attach this export and set RESUME_INPUT to its root. Reattach all three immutable source basis datasets.\n")
files = sorted(path for path in EXPORT_ROOT.rglob("*") if path.is_file())
(EXPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.relative_to(EXPORT_ROOT).as_posix()}" for path in files) + "\n")
print("Export root:", EXPORT_ROOT, "files:", len(files))
if UPLOAD_AS_KAGGLE_DATASET:
    import kagglehub
    kagglehub.dataset_upload(KAGGLE_DATASET_HANDLE, str(EXPORT_ROOT), version_notes=f"Frozen answer-colon inference ablation at {commit}")